In [15]:
import random
import nltk
from nltk import Tree
import matplotlib.pyplot as plt
from collections import defaultdict
import copy

In [16]:
# Task 1: Load and Represent the CFG

class CFG:
    def __init__(self):
        self.rules = {
            'S': [['NP', 'VP']],
            'NP': [['Det', 'N'], ['Det', 'Adj', 'N'], ['NP', 'PP'], ['NP', 'RelClause']],
            'RelClause': [['RelPro', 'VP']],
            'VP': [['V', 'NP'], ['V'], ['VP', 'PP']],
            'PP': [['Prep', 'NP']],
            'Det': [['the'], ['a']],
            'N': [['boy'], ['girl'], ['dog'], ['telescope'], ['man'], ['park']],
            'Adj': [['tall'], ['young'], ['old']],
            'RelPro': [['who'], ['that']],
            'V': [['saw'], ['chased'], ['liked'], ['ate']],
            'Prep': [['with'], ['in'], ['near']]
        }
    
    def display_rules(self):
        print("Context-Free Grammar Rules:")
        print("=" * 30)
        for lhs, productions in self.rules.items():
            for production in productions:
                print(f"{lhs} → {' '.join(production)}")

# Initialize and display grammar
cfg = CFG()
cfg.display_rules()

Context-Free Grammar Rules:
S → NP VP
NP → Det N
NP → Det Adj N
NP → NP PP
NP → NP RelClause
RelClause → RelPro VP
VP → V NP
VP → V
VP → VP PP
PP → Prep NP
Det → the
Det → a
N → boy
N → girl
N → dog
N → telescope
N → man
N → park
Adj → tall
Adj → young
Adj → old
RelPro → who
RelPro → that
V → saw
V → chased
V → liked
V → ate
Prep → with
Prep → in
Prep → near


In [17]:
# Task 2: Generate Valid Sentences

class SentenceGenerator:
    def __init__(self, cfg):
        self.cfg = cfg
        
    def generate(self, symbol, depth=0, max_depth=10):
        if depth > max_depth:
            return []
        
        if symbol in self.cfg.rules:
            productions = self.cfg.rules[symbol]
            chosen_production = random.choice(productions)
            result = []
            for sym in chosen_production:
                result.extend(self.generate(sym, depth + 1, max_depth))
            return result
        else:
            return [symbol]
    
    def generate_sentences(self, count=10):
        sentences = set()
        attempts = 0
        while len(sentences) < count and attempts < 100:
            words = self.generate('S')
            if words:
                sentence = ' '.join(words)
                if len(sentence.split()) <= 15:
                    sentences.add(sentence)
            attempts += 1
        return list(sentences)
    
    def check_requirements(self, sentences):
        rel_clause_count = 0
        pp_count = 0
        rel_words = ['who', 'that']
        prep_words = ['with', 'in', 'near']
        
        for sentence in sentences:
            if any(word in sentence for word in rel_words):
                rel_clause_count += 1
            if any(word in sentence for word in prep_words):
                pp_count += 1
        
        return rel_clause_count, pp_count

generator = SentenceGenerator(cfg)
generated_sentences = generator.generate_sentences(10)

print("Generated Sentences:")
print("=" * 20)
for i, sentence in enumerate(generated_sentences, 1):
    print(f"{i:2d}. {sentence}")

rel_count, pp_count = generator.check_requirements(generated_sentences)
print(f"\nRelative clauses: {rel_count}, Prepositional phrases: {pp_count}")

Generated Sentences:
 1. a dog near the young park saw
 2. the dog liked the girl
 3. a young park who ate the boy in a boy who saw in who saw
 4. a old boy chased
 5. the girl ate in the boy
 6. the tall dog with a girl in a young boy saw a tall boy
 7. a old dog ate
 8. the young boy saw the man with the man
 9. a park chased the park
10. a old man saw a tall man

Relative clauses: 1, Prepositional phrases: 5


In [18]:
# Task 3: Top-Down Parsing (Recursive Descent)

class TopDownParser:
    def __init__(self, cfg):
        self.cfg = cfg
        self.derivation = []
        
    def parse(self, sentence):
        self.words = sentence.split()
        self.position = 0
        self.derivation = []
        
        print(f"Parsing: {sentence}")
        print("=" * 40)
        
        result = self.parse_NP_VP()  # Direct parsing for S → NP VP
        if result and self.position == len(self.words):
            print("\nStep-by-step derivation:")
            for step in self.derivation:
                print(step)
            return result
        else:
            print("Parse failed")
            return None
    
    def parse_NP_VP(self):
        self.derivation.append("S → NP VP")
        saved_pos = self.position
        
        np = self.parse_NP()
        if np is None:
            self.position = saved_pos
            return None
            
        vp = self.parse_VP()
        if vp is None:
            self.position = saved_pos
            return None
            
        return Tree('S', [np, vp])
    
    def parse_NP(self):
        saved_pos = self.position
        
        # Try NP → Det N
        det = self.parse_terminal(['the', 'a'])
        if det:
            n = self.parse_terminal(['boy', 'girl', 'dog', 'telescope', 'man', 'park'])
            if n:
                self.derivation.append("NP → Det N")
                return Tree('NP', [det, n])
        
        self.position = saved_pos
        
        # Try NP → Det Adj N
        det = self.parse_terminal(['the', 'a'])
        if det:
            adj = self.parse_terminal(['tall', 'young', 'old'])
            if adj:
                n = self.parse_terminal(['boy', 'girl', 'dog', 'telescope', 'man', 'park'])
                if n:
                    self.derivation.append("NP → Det Adj N")
                    return Tree('NP', [det, adj, n])
        
        self.position = saved_pos
        return None
    
    def parse_VP(self):
        saved_pos = self.position
        
        # Try VP → V NP
        v = self.parse_terminal(['saw', 'chased', 'liked', 'ate'])
        if v:
            np = self.parse_NP()
            if np:
                self.derivation.append("VP → V NP")
                return Tree('VP', [v, np])
        
        self.position = saved_pos
        
        # Try VP → V
        v = self.parse_terminal(['saw', 'chased', 'liked', 'ate'])
        if v:
            self.derivation.append("VP → V")
            return Tree('VP', [v])
        
        self.position = saved_pos
        return None
    
    def parse_terminal(self, terminals):
        if self.position < len(self.words) and self.words[self.position] in terminals:
            word = self.words[self.position]
            self.position += 1
            # Create appropriate non-terminal based on the word
            if word in ['the', 'a']:
                return Tree('Det', [word])
            elif word in ['boy', 'girl', 'dog', 'telescope', 'man', 'park']:
                return Tree('N', [word])
            elif word in ['tall', 'young', 'old']:
                return Tree('Adj', [word])
            elif word in ['saw', 'chased', 'liked', 'ate']:
                return Tree('V', [word])
            elif word in ['with', 'in', 'near']:
                return Tree('Prep', [word])
            elif word in ['who', 'that']:
                return Tree('RelPro', [word])
        return None

parser = TopDownParser(cfg)

# Test sentences
test_sentences = [
    "the boy saw the dog",
    "a tall girl chased the boy who liked the dog",
    "the man ate with the telescope"
]

parse_trees = []
for sentence in test_sentences:
    tree = parser.parse(sentence)
    if tree:
        print(f"\nParse tree for '{sentence}':")
        tree.pretty_print()
        parse_trees.append((sentence, tree))
    print("\n" + "="*50 + "\n")

Parsing: the boy saw the dog

Step-by-step derivation:
S → NP VP
NP → Det N
NP → Det N
VP → V NP

Parse tree for 'the boy saw the dog':
             S             
      _______|___           
     |           VP        
     |        ___|___       
     NP      |       NP    
  ___|___    |    ___|___   
Det      N   V  Det      N 
 |       |   |   |       |  
the     boy saw the     dog



Parsing: a tall girl chased the boy who liked the dog
Parse failed


Parsing: the man ate with the telescope
Parse failed




In [19]:
# Task 4: Bottom-Up Parsing (Shift-Reduce)

class ShiftReduceParser:
    def __init__(self, cfg):
        self.cfg = cfg
        self.create_reduction_table()
    
    def create_reduction_table(self):
        self.reductions = {}
        # Create reduction table for non-left-recursive rules
        simple_rules = {
            'Det': ['the', 'a'],
            'N': ['boy', 'girl', 'dog', 'telescope', 'man', 'park'],
            'Adj': ['tall', 'young', 'old'],
            'V': ['saw', 'chased', 'liked', 'ate'],
            'Prep': ['with', 'in', 'near'],
            'RelPro': ['who', 'that']
        }
        
        # Add terminal reductions
        for lhs, terminals in simple_rules.items():
            for terminal in terminals:
                self.reductions[terminal] = [lhs]
        
        # Add phrase structure rules
        self.reductions['Det N'] = ['NP']
        self.reductions['Det Adj N'] = ['NP']
        self.reductions['V'] = ['VP']
        self.reductions['V NP'] = ['VP']
        self.reductions['Prep NP'] = ['PP']
        self.reductions['NP VP'] = ['S']
    
    def parse(self, sentence):
        words = sentence.split()
        stack = []
        input_buffer = words[:]
        
        print(f"Parsing: {sentence}")
        print("=" * 50)
        print(f"{'Step':4} {'Action':20} {'Stack':25} {'Input'}")
        print("-" * 50)
        
        step = 0
        
        while input_buffer or len(stack) > 1:
            step += 1
            
            # Try to reduce
            reduced = False
            
            # Check for possible reductions from longest to shortest
            for length in range(min(3, len(stack)), 0, -1):
                if len(stack) >= length:
                    substring = ' '.join(stack[-length:])
                    if substring in self.reductions:
                        reduction = self.reductions[substring][0]
                        stack = stack[:-length] + [reduction]
                        action = f"REDUCE {substring} → {reduction}"
                        print(f"{step:4} {action:20} {' '.join(stack):25} {' '.join(input_buffer)}")
                        reduced = True
                        break
            
            if not reduced:
                if input_buffer:
                    # Shift
                    token = input_buffer.pop(0)
                    stack.append(token)
                    action = f"SHIFT {token}"
                    print(f"{step:4} {action:20} {' '.join(stack):25} {' '.join(input_buffer)}")
                else:
                    break
            
            # Check for acceptance
            if len(stack) == 1 and stack[0] == 'S' and not input_buffer:
                print(f"{step+1:4} {'ACCEPT':20}")
                return True
        
        if len(stack) == 1 and stack[0] == 'S' and not input_buffer:
            print(f"{step+1:4} {'ACCEPT':20}")
            return True
        else:
            print(f"{step+1:4} {'REJECT':20}")
            return False

sr_parser = ShiftReduceParser(cfg)

for sentence in test_sentences:
    result = sr_parser.parse(sentence)
    print(f"Result: {'ACCEPTED' if result else 'REJECTED'}")
    print("\n" + "="*50 + "\n")

Parsing: the boy saw the dog
Step Action               Stack                     Input
--------------------------------------------------
   1 SHIFT the            the                       boy saw the dog
   2 REDUCE the → Det     Det                       boy saw the dog
   3 SHIFT boy            Det boy                   saw the dog
   4 REDUCE boy → N       Det N                     saw the dog
   5 REDUCE Det N → NP    NP                        saw the dog
   6 SHIFT saw            NP saw                    the dog
   7 REDUCE saw → V       NP V                      the dog
   8 REDUCE V → VP        NP VP                     the dog
   9 REDUCE NP VP → S     S                         the dog
  10 SHIFT the            S the                     dog
  11 REDUCE the → Det     S Det                     dog
  12 SHIFT dog            S Det dog                 
  13 REDUCE dog → N       S Det N                   
  14 REDUCE Det N → NP    S NP                      
  16 REJECT            

In [20]:
# Task 5: Ambiguity Detection

class AmbiguityParser:
    def __init__(self, cfg):
        self.cfg = cfg
        
    def find_all_parses(self, sentence):
        words = sentence.split()
        self.words = words
        self.memo = {}
        trees = self.parse_with_cky('S', 0, len(words))
        return trees if trees else []
    
    def parse_with_cky(self, symbol, start, end):
        if (symbol, start, end) in self.memo:
            return self.memo[(symbol, start, end)]
        
        trees = []
        
        if symbol in self.cfg.rules:
            for production in self.cfg.rules[symbol]:
                if len(production) == 1:
                    # Terminal production
                    term = production[0]
                    if (start + 1 == end and start < len(self.words) and 
                        self.words[start] == term):
                        trees.append(Tree(symbol, [Tree(term, [])]))
                
                elif len(production) == 2:
                    # Binary production
                    left_sym, right_sym = production
                    for split in range(start + 1, end):
                        left_trees = self.parse_with_cky(left_sym, start, split)
                        right_trees = self.parse_with_cky(right_sym, split, end)
                        
                        for left in left_trees:
                            for right in right_trees:
                                trees.append(Tree(symbol, [left, right]))
                
                elif len(production) == 3:
                    # Ternary production
                    first_sym, second_sym, third_sym = production
                    for split1 in range(start + 1, end - 1):
                        for split2 in range(split1 + 1, end):
                            first_trees = self.parse_with_cky(first_sym, start, split1)
                            second_trees = self.parse_with_cky(second_sym, split1, split2)
                            third_trees = self.parse_with_cky(third_sym, split2, end)
                            
                            for first in first_trees:
                                for second in second_trees:
                                    for third in third_trees:
                                        trees.append(Tree(symbol, [first, second, third]))
        
        self.memo[(symbol, start, end)] = trees
        return trees

ambiguity_sentence = "the man saw the boy with the telescope"
amb_parser = AmbiguityParser(cfg)
all_parses = amb_parser.find_all_parses(ambiguity_sentence)

print(f"Ambiguous sentence: '{ambiguity_sentence}'")
print(f"Number of possible parse trees: {len(all_parses)}")

if len(all_parses) > 0:
    print("\nAll possible parse trees:")
    print("=" * 40)
    
    for i, tree in enumerate(all_parses, 1):
        print(f"\nParse Tree {i}:")
        tree.pretty_print()
        print("-" * 40)
else:
    print("No valid parse trees found for this sentence.")

Ambiguous sentence: 'the man saw the boy with the telescope'
Number of possible parse trees: 2

All possible parse trees:

Parse Tree 1:
                 S                                
      ___________|_______                          
     |                   VP                       
     |        ___________|___                      
     |       |               NP                   
     |       |        _______|____                 
     |       |       |            PP              
     |       |       |        ____|___             
     NP      |       NP      |        NP          
  ___|___    |    ___|___    |     ___|______      
Det      N   V  Det      N  Prep Det         N    
 |       |   |   |       |   |    |          |     
the     man saw the     boy with the     telescope
 |       |   |   |       |   |    |          |     
...     ... ... ...     ... ...  ...        ...   

----------------------------------------

Parse Tree 2:
                 S                

In [21]:
import nltk
from nltk.tree import Tree

def create_simple_tree_visualization():
    try:
        nltk.download('punkt', quiet=True)
        print("NLTK Tree Visualizations:")
        print("=" * 30)
        
        sample_tree = Tree('S', [
            Tree('NP', [Tree('Det', ['the']), Tree('N', ['boy'])]),
            Tree('VP', [Tree('V', ['saw']), 
                       Tree('NP', [Tree('Det', ['the']), Tree('N', ['dog'])])])
        ])
        
        print("Sample Parse Tree:")
        sample_tree.pretty_print()

        # DO NOT CALL sample_tree.draw()

    except Exception as e:
        print(f"Visualization error: {e}")


def ascii_tree_printer(tree, prefix="", is_last=True):
    if isinstance(tree, Tree):
        connector = "└── " if is_last else "├── "
        print(f"{prefix}{connector}{tree.label()}")
        
        children = list(tree)
        for i, child in enumerate(children):
            is_last_child = (i == len(children) - 1)
            extension = "    " if is_last else "│   "
            ascii_tree_printer(child, prefix + extension, is_last_child)
    else:
        connector = "└── " if is_last else "├── "
        print(f"{prefix}{connector}'{tree}'")


# Example Tree
print("\nASCII Tree Representations:")
print("=" * 30)

example_tree = Tree('S', [
    Tree('NP', [Tree('Det', ['the']), Tree('Adj', ['tall']), Tree('N', ['boy'])]),
    Tree('VP', [
        Tree('V', ['saw']),
        Tree('NP', [
            Tree('NP', [Tree('Det', ['the']), Tree('N', ['girl'])]),
            Tree('PP', [Tree('Prep', ['with']), 
                        Tree('NP', [Tree('Det', ['the']), Tree('N', ['telescope'])])])
        ])
    ])
])

ascii_tree_printer(example_tree)
create_simple_tree_visualization()



ASCII Tree Representations:
└── S
    ├── NP
    │   ├── Det
    │   │   └── 'the'
    │   ├── Adj
    │   │   └── 'tall'
    │   └── N
    │       └── 'boy'
    └── VP
        ├── V
        │   └── 'saw'
        └── NP
            ├── NP
            │   ├── Det
            │   │   └── 'the'
            │   └── N
            │       └── 'girl'
            └── PP
                ├── Prep
                │   └── 'with'
                └── NP
                    ├── Det
                    │   └── 'the'
                    └── N
                        └── 'telescope'
NLTK Tree Visualizations:
Sample Parse Tree:
             S             
      _______|___           
     |           VP        
     |        ___|___       
     NP      |       NP    
  ___|___    |    ___|___   
Det      N   V  Det      N 
 |       |   |   |       |  
the     boy saw the     dog

